# Explorer les commandes

Avant de tracer, identifier la ligne observée, vérifier les colonnes et comprendre ce que recouvre une commande.

**Question métier :** Que représente une ligne du fichier, et combien de commandes distinctes contient-il ?

Les données sont montées en lecture seule. Enregistrez vos figures dans `/workspace/resultats`.

In [ ]:
# %pip install xlrd
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_excel(
    "/workspace/data/sample_-_superstore.xls",
    sheet_name="Orders"
)

df.info()

In [ ]:
orders = df
orders['Order Date'] = pd.to_datetime(orders['Order Date'])

## Notion 1 — Charger et décrire la feuille Orders


### Étape 1 — Contrôler le tableau

Vérifiez les dimensions et les colonnes après le chargement du classeur.

In [ ]:
print(orders.shape)
print(orders.columns.tolist())

### Étape 2 — Compter les observations

Comparez le nombre de lignes au nombre d’identifiants de commande distincts.

In [ ]:
print('Lignes :', len(orders))
print('Commandes distinctes :', orders['Order ID'].nunique())
print('Catégories :', orders['Category'].nunique())

### Étape 3 — Interpréter

La feuille contient 10 194 lignes pour 5 111 commandes distinctes. Une ligne décrit une ligne de commande ; un même Order ID peut se répéter.

### Exercice 1 — Compter les sous-catégories distinctes

Combien de sous-catégories différentes sont présentes dans la feuille Orders ?

**À rendre :** Le nombre de sous-catégories distinctes et la ligne de code utilisée.

In [ ]:
nombre_sous_categories = orders['Sub-Category'].nunique()
print('Sous-catégories distinctes :', nombre_sous_categories)

### Mise en commun

La feuille contient 17 sous-catégories distinctes. nunique() compte les valeurs différentes de la colonne Sub-Category.

## Notion 2 — Comparer lignes et commandes par catégorie


### Étape 1 — Calculer les deux comptages

Pour chaque catégorie, comptez les lignes de commande et les identifiants Order ID distincts.

In [ ]:
# Regrouper les lignes par catégorie et compter les commandes
par_categorie = orders.groupby("Category")["Order ID"].agg(
    lignes="size",       # Nombre de lignes de commande
    commandes="nunique"  # Nombre de commandes distinctes
)

# Afficher les deux comptages par catégorie
print(par_categorie)

### Étape 2 — Construire les barres groupées

Les deux séries partagent le même axe. Vert et lime distinguent les lignes des commandes distinctes, dont les valeurs sont affichées au bout des barres.

In [ ]:
# Création d'une figure et de ses axes
# figsize définit la largeur et la hauteur en pouces
fig, ax = plt.subplots(figsize=(9, 4))

# Positions verticales des catégories : 0, 1, 2...
y = range(len(par_categorie))

# Première série : nombre de lignes par catégorie
# +0.18 décale les barres vers le haut
# height définit l'épaisseur des barres
b1 = ax.barh(
    [i + 0.18 for i in y],
    par_categorie["lignes"],
    height=0.36,
    label="Lignes",
    color="#275c47"
)

# Deuxième série : nombre de commandes distinctes
# -0.18 décale les barres vers le bas
b2 = ax.barh(
    [i - 0.18 for i in y],
    par_categorie["commandes"],
    height=0.36,
    label="Commandes distinctes",
    color="#b5cf66"
)

# Affichage des valeurs numériques au bout des barres
# padding=3 définit un espacement de 3 points
ax.bar_label(b1, padding=3)
ax.bar_label(b2, padding=3)

# Remplacement des positions 0, 1, 2...
# par les véritables noms des catégories
ax.set_yticks(list(y), par_categorie.index)

# Titre du graphique et nom de l'axe horizontal
ax.set(
    title="Lignes et commandes par catégorie",
    xlabel="Nombre"
)

# Affichage de la légende
ax.legend()

# Ajout d'une marge horizontale de 12 %
# pour laisser de la place aux valeurs affichées
ax.margins(x=0.12)

# Ajustement automatique des espacements
plt.tight_layout()

# Affichage du graphique
plt.show()

### Étape 3 — Interpréter

Office Supplies compte le plus de lignes et de commandes distinctes dans cette ventilation. Les commandes peuvent apparaître dans plusieurs catégories : les comptes par catégorie ne s’additionnent donc pas au total général.

### Exercice 2 — Comparer les écarts par région

Dans quelle région l’écart entre lignes et commandes distinctes est-il le plus grand ?

**À rendre :** La région, les deux comptages, l’écart et une phrase qui interprète cet écart.

In [ ]:
par_region = orders.groupby('Region').agg(lignes=('Order ID','size'), commandes=('Order ID','nunique'))
par_region['ecart'] = par_region['lignes'] - par_region['commandes']
print(par_region.loc[par_region['ecart'].idxmax()])

### Mise en commun

West a le plus grand écart : 3 253 lignes contre 1 635 commandes distinctes, soit 1 618 lignes de plus. Plusieurs lignes peuvent appartenir à une même commande.

## Notion 3 — Suivre le nombre de commandes par année


### Étape 1 — Extraire l’année et compter les commandes

Créez une colonne année, puis comptez les identifiants Order ID distincts pour chaque année.

In [ ]:
# Ajouter l'année de chaque commande
orders["annee"] = orders["Order Date"].dt.year

# Compter les commandes distinctes par année
commandes_annee = orders.groupby("annee")["Order ID"].nunique()

print(commandes_annee)

### Étape 2 — Tracer le résultat

Les barres comparent le nombre de commandes distinctes par année.

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(commandes_annee.index.astype(str), commandes_annee.values)
ax.set(title='Commandes distinctes par année', xlabel='Nombre de commandes')
plt.show()

### Étape 3 — Interpréter

Le nombre de commandes distinctes augmente dans le fichier au fil des années. Ce constat descriptif ne suffit pas à attribuer une cause à cette évolution.

### Exercice 3 — Comparer les clients par année

Comment le nombre de clients distincts évolue-t-il selon les années ? Représentez le résultat.

**À rendre :** Un graphique annuel, les quatre effectifs de clients et une phrase de constat.

In [ ]:
clients_annee = orders.groupby("annee")["Customer ID"].nunique()
print(clients_annee)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(clients_annee.index.astype(str), clients_annee.values, color="#275c47")
ax.set(title="Clients distincts par année", xlabel="Année", ylabel="Nombre de clients")
plt.tight_layout()
plt.show()

### Mise en commun

Les effectifs sont 605 en 2023, 577 en 2024, 653 en 2025 et 704 en 2026. Un même client peut être compté dans plusieurs années ; les effectifs annuels ne s’additionnent pas pour compter les clients uniques du fichier.